In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-lgd-starting-feats-2'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

boto3==1.24.59
pandas==1.2.4
scikit_learn==0.24.1
tqdm==4.64.1

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import numpy as np
import json
import boto3
import time
from tqdm import tqdm

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_model = '03_pricing_lgd'
    
    # get df_hyperparameters
    str_filename = 'df_hyperparameters.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/12_step_function/{str_filename}'
    df = pd.read_csv(str_uri)
    # convert to dict
    dict_hyperparameters = dict(zip(df['keys'], df['values']))
    
    # get eval metric
    str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
    print(f'Eval metric: {str_eval_metric}')
        
    # load output from shared feature selection
    print('Loading output from shared feature selection...')
    str_filename = 'df_iterative_feat_select.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/01_feat_select/04_batch_feature_selection/results/{str_filename}'
    df_feat_select = pd.read_csv(str_uri)
    
    # get max n feats and remove it
    print('Removing max n feats...')
    int_max_feats = np.max(df_feat_select['n_feats'])
    df_feat_select = df_feat_select[df_feat_select['n_feats'] < int_max_feats]
    
    # logic for sorting
    print('Sorting...')
    if str_eval_metric in ['AUC','PRAUC','F1']:
        bool_ascending = False
    else:
        bool_ascending = True # works for RMSE
    df_feat_select.sort_values(by='flt_eval_metric_valid', ascending=bool_ascending, inplace=True)

    # get list of best features
    list_cols_model = eval(df_feat_select['list_cols_model'].iloc[0])
    
    # rm down cash and down total
    print('Removing down cash and down total...')
    list_cols_down = [
        'fltdowncash__app',
        'fltapproveddowntotal__app',
        'jt41s__tu',
    ]
    list_cols_model = [col for col in list_cols_model if col not in list_cols_down]
    
    # load in the list of features to drop
    print('Loading list of features to drop...')
    str_filename = 'df_feats_to_drop.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    try:
        list_feats_drop = list(pd.read_csv(str_uri)['feature'])
    except:
        # create df_feats_to_drop
        list_feats_drop = []
        df_feats_to_drop = pd.DataFrame({'feature': list_feats_drop})
        # write
        df_feats_to_drop.to_csv(str_uri, index=False)
    
    # remove the list of features to drop
    print('Removing the list of features to drop...')
    list_cols_model = [col for col in list_cols_model if col not in list_feats_drop]
    
    # write to s3
    print('Writing list of columns in model to s3...')
    df_cols_in_model = pd.DataFrame({'feature': list_cols_model})
    str_filename = 'df_cols_in_model.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    df_cols_in_model.to_csv(str_uri, index=False)
    
    # write to s3 as json for map in step function
    print('Writing json of columns in model to s3...')
    str_list_cols_model = json.dumps(list_cols_model)
    cls_client_s3 = boto3.client('s3')
    str_filename = 'json_cols_in_model.json'
    str_key = f'{str_model}/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    cls_client_s3.put_object(
        Bucket=str_project,
        Key=str_key,
        Body=str_list_cols_model,
    )

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-lgd-starting-feats-2

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  43.52kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 303d20c679a3
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> d25abc48fcf6
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> b38f093690ef
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> 2f53bab4bad1
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> Using cache
 ---> 6c4176f4b1bb
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Using cache
 ---> 29119c5352db
Successfully built 29119c5352db
Successfully tagged genxii-lgd-starting-feats-2:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-lgd-starting-feats-2' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-lgd-starting-feats-2]
e7db69de5818: Preparing
d0f9f4244c48: Preparing
470ffb2c4383: Preparing
d630e2305053: Preparing
3bd433acfe84: Preparing
09b55d38856d: Preparing
e073f5919ae5: Preparing
b3b414f01759: Preparing
8308f08f35ba: Preparing
c8203e562a8c: Preparing
09b55d38856d: Waiting
e073f5919ae5: Waiting
b3b414f01759: Waiting
8308f08f35ba: Waiting
c8203e562a8c: Waiting
e7db69de5818: Layer already exists
3bd433acfe84: Layer already exists
d0f9f4244c48: Layer already exists
470ffb2c4383: Layer already exists
d630e2305053: Layer already exists
e073f5919ae5: Layer already exists
09b55d38856d: Layer already exists
b3b414f01759: Layer already exists
8308f08f35ba: Layer already exists
c8203e562a8c: Layer already exists
latest: digest: sha256:b798394c76ff4505a53cf3a239b8654bf62a9381b5a1ecaacf2fd2245832360a size: 2420


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 20 Aug 2024 15:56:17 GMT',
                                      'x-amzn-requestid': '934fa42f-e5ca-4a8c-b04c-0f930c2cc211'},
                      'HTTPStatusCode': 204,
                      'RequestId': '934fa42f-e5ca-4a8c-b04c-0f930c2cc211',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': 'b798394c76ff4505a53cf3a239b8654bf62a9381b5a1ecaacf2fd2245832360a',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-starting-feats-2',
 'FunctionName': 'genxii-lgd-starting-feats-2',
 'LastModified': '2024-08-20T15:56:17.094+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-lgd-starting-feats-2'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1213',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 20 Aug 2024 15:56:17 GMT',
                                      'x-amzn-requestid': 'a6a1ac6b-c2e2-4c98-994d-4b846ad2bc78'},
                      'HTTPStatusCode': 201,
                      'RequestId': 'a

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)